In [1]:
import sys

sys.path.append("../scripts")

In [2]:
import pandas as pd
import numpy as np
import preprocessing

In [3]:
data = pd.read_csv('../data/car_price_prediction.csv')
data.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [4]:
data = data.astype('object')
data = preprocessing.preprocessing_pipeline(data)

Preprocessing started...
initial shape: (19237, 18)
After dropping duplicates: (18924, 18)
Replacing categorical values with numerical...
After cleaning outliers: (16312, 18)
Feature engineering...
Droping unnecessary columns...
Final shape after preprocessing: (16312, 16)


In [5]:
data.head()

,Price,Levy,Manufacturer,Model,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Wheel,Color,Airbags,Age
0,13328,1399,LEXUS,RX 450,Jeep,Yes,Hybrid,3.5,186005,6.0,Automatic,4x4,Left wheel,Silver,12,16
1,16621,1018,CHEVROLET,Equinox,Jeep,No,Petrol,3.0,192000,6.0,Tiptronic,4x4,Left wheel,Black,8,15
2,8467,0,HONDA,FIT,Hatchback,No,Petrol,1.3,200000,4.0,Variator,Front,Right-hand drive,Black,2,20
3,3607,862,FORD,Escape,Jeep,Yes,Hybrid,2.5,168966,4.0,Automatic,4x4,Left wheel,White,0,15
4,11726,446,HONDA,FIT,Hatchback,Yes,Petrol,1.3,91901,4.0,Automatic,Front,Left wheel,Silver,4,12


In [6]:
from sklearn.preprocessing import LabelEncoder,StandardScaler

one_hot_columns = ['Leather interior', 'Gear box type', 'Drive wheels', 'Wheel']

data = pd.get_dummies(data, columns=one_hot_columns)

label_encode_columns = ['Fuel type', 'Category', 'Color']

label_encoder = LabelEncoder()

for column in label_encode_columns:
    data[column] = label_encoder.fit_transform(data[column])

In [7]:
data

,Price,Levy,Manufacturer,Model,Category,Fuel type,Engine volume,Mileage,Cylinders,Color,...,Leather interior_Yes,Gear box type_Automatic,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator,Drive wheels_4x4,Drive wheels_Front,Drive wheels_Rear,Wheel_Left wheel,Wheel_Right-hand drive
0,13328,1399,LEXUS,RX 450,4,2,3.5,186005,6.0,12,...,True,True,False,False,False,True,False,False,True,False
1,16621,1018,CHEVROLET,Equinox,4,5,3.0,192000,6.0,1,...,False,False,False,True,False,True,False,False,True,False
2,8467,0,HONDA,FIT,3,5,1.3,200000,4.0,1,...,False,False,False,False,True,False,True,False,False,True
3,3607,862,FORD,Escape,4,2,2.5,168966,4.0,14,...,True,True,False,False,False,True,False,False,True,False
4,11726,446,HONDA,FIT,3,5,1.3,91901,4.0,12,...,True,True,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19232,8467,0,MERCEDES-BENZ,CLK 200,1,0,2.0,300000,4.0,12,...,True,False,True,False,False,False,False,True,True,False
19233,15681,831,HYUNDAI,Sonata,9,5,2.4,161600,4.0,11,...,True,False,False,True,False,False,True,False,True,False
19234,26108,836,HYUNDAI,Tucson,4,1,2.0,116365,4.0,7,...,True,True,False,False,False,False,True,False,True,False
19235,5331,1288,CHEVROLET,Captiva,4,1,2.0,51258,4.0,1,...,True,True,False,False,False,False,True,False,True,False


In [8]:
X = data.drop('Price', axis=1)
y = data['Price']

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f'Train set: {len(X_train)} samples')
print(f'Validation set: {len(X_val)} samples')
print(f'Test set: {len(X_test)} samples')

Train set: 11418 samples
Validation set: 2447 samples
Test set: 2447 samples


In [10]:
target_encode_columns = ['Fuel type', 'Model', 'Airbags', 'Cylinders', 'Manufacturer']

train = pd.concat([X_train, y_train], axis=1)
for col in target_encode_columns:
    mean_encoded = train.groupby(col)['Price'].mean()
    global_mean = train['Price'].mean()
    X_train[col] = X_train[col].map(mean_encoded).fillna(global_mean)
    X_val[col] = X_val[col].map(mean_encoded).fillna(global_mean)
    X_test[col] = X_test[col].map(mean_encoded).fillna(global_mean)

In [11]:
X_train

,Levy,Manufacturer,Model,Category,Fuel type,Engine volume,Mileage,Cylinders,Color,Airbags,...,Leather interior_Yes,Gear box type_Automatic,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator,Drive wheels_4x4,Drive wheels_Front,Drive wheels_Rear,Wheel_Left wheel,Wheel_Right-hand drive
14446,629,9502.40625,10252.768421,4,13516.229323,1.6,103358,14797.498735,2,8886.49058,...,True,True,False,False,False,True,False,False,True,False
8010,1327,13153.311819,11782.534846,9,13516.229323,2.5,56173,14797.498735,12,10090.820419,...,True,True,False,False,False,False,True,False,True,False
10003,1195,12445.307463,13844.224,6,20472.352273,2.2,182320,14797.498735,14,19595.851594,...,False,False,True,False,False,False,True,False,True,False
16052,753,8714.291667,4446.571429,9,13516.229323,2.4,110000,14797.498735,2,17039.637236,...,False,False,False,True,False,False,True,False,True,False
1553,0,12627.019349,15424.210526,9,20472.352273,2.2,280000,14797.498735,2,9161.857143,...,True,False,False,True,False,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15825,0,13822.281027,20123.333333,1,13516.229323,3.6,131000,12165.99358,11,14134.342331,...,False,False,False,True,False,False,False,True,True,False
6331,781,13153.311819,11782.534846,9,10451.618482,2.5,151195,14797.498735,7,10090.820419,...,True,True,False,False,False,False,True,False,True,False
998,0,10882.89372,7363.4,7,13516.229323,1.7,0,14797.498735,12,19595.851594,...,False,True,False,False,False,False,True,False,False,True
18633,1172,10730.359281,7786.990566,4,10451.618482,3.5,90000,12165.99358,12,10090.820419,...,True,True,False,False,False,True,False,False,True,False


In [12]:
numerical_columns = ['Levy', 'Engine volume', 'Mileage', 'Age', 'Fuel type', 'Model', 'Airbags', 'Cylinders', 'Manufacturer']

scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [13]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](22,)","[-1323. , -218.05, 5172.87,..., 497.4 , 1203.09,-1203.09]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](22,)","['Levy','Manufacturer','Model',...,'Drive wheels_Rear','Wheel_Left wheel', 'Wheel_Right-hand drive']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.689e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,22
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(18)


In [14]:
from sklearn.metrics import root_mean_squared_error, r2_score

y_val_pred = lr.predict(X_val)
rmse = root_mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print(f'Validation RMSE: {rmse}')
print(f'Validation R2: {r2}')

Validation RMSE: 8325.849078040024
Validation R2: 0.4808126056647227


In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

rf = RandomForestRegressor()
rf.fit(X_train, y_train)

y_val_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print(f'Validation RMSE: {rmse}')
print(f'Validation R2: {r2}')

Validation RMSE: 5563.081329713672
Validation R2: 0.7682082953664864
